# Revenue Analysis and Customer-Level

## Libraries

In [1]:
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as st
from datetime import datetime, timedelta
from IPython.display import display

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(40)

BLUE, ORANGE = '#2E86AB', '#F26419'
ALPHA = 0.05

print('All libraries are ready to use.')

All libraries are ready to use.


## Data loading

In [2]:
raw = pl.read_csv(
    'data.csv',
    schema_overrides={'InvoiceNo': pl.String, 'StockCode': pl.String},
    encoding='utf8-lossy'
)

In [3]:
raw

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,i64,str,f64,i64,str
"""536365""","""85123A""","""WHITE HANGING HEART T-LIGHT HOLDER""",6,"""12/1/2010 8:26""",2.55,17850,"""United Kingdom"""
"""536365""","""71053""","""WHITE METAL LANTERN""",6,"""12/1/2010 8:26""",3.39,17850,"""United Kingdom"""
"""536365""","""84406B""","""CREAM CUPID HEARTS COAT HANGER""",8,"""12/1/2010 8:26""",2.75,17850,"""United Kingdom"""
"""536365""","""84029G""","""KNITTED UNION FLAG HOT WATER BOTTLE""",6,"""12/1/2010 8:26""",3.39,17850,"""United Kingdom"""
"""536365""","""84029E""","""RED WOOLLY HOTTIE WHITE HEART.""",6,"""12/1/2010 8:26""",3.39,17850,"""United Kingdom"""
"""536365""","""22752""","""SET 7 BABUSHKA NESTING BOXES""",2,"""12/1/2010 8:26""",7.65,17850,"""United Kingdom"""
"""536365""","""21730""","""GLASS STAR FROSTED T-LIGHT HOLDER""",6,"""12/1/2010 8:26""",4.25,17850,"""United Kingdom"""
"""536366""","""22633""","""HAND WARMER UNION JACK""",6,"""12/1/2010 8:28""",1.85,17850,"""United Kingdom"""
"""536366""","""22632""","""HAND WARMER RED POLKA DOT""",6,"""12/1/2010 8:28""",1.85,17850,"""United Kingdom"""


## 1. Data Overview and Cleaning

### 1.1 Overview

In [4]:
print(f'Rows x columns : {raw.shape[0]:,} x {raw.shape[1]}')
print(f"Date format    : {raw['InvoiceDate'][0]}  (month/day/year hour:minute, stored as text)")
display(raw.describe())

Rows x columns : 541,909 x 8
Date format    : 12/1/2010 8:26  (month/day/year hour:minute, stored as text)


statistic,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,str,f64,str,f64,f64,str
"""count""","""541909""","""541909""","""540455""",541909.0,"""541909""",541909.0,406829.0,"""541909"""
"""null_count""","""0""","""0""","""1454""",0.0,"""0""",0.0,135080.0,"""0"""
"""mean""",null,null,null,9.55225,null,4.611114,15287.69057,null
"""std""",null,null,null,218.081158,null,96.759853,1713.600303,null
"""min""","""536365""","""10002""",""" 4 PURPLE FLOCK DINNER CANDLES""",-80995.0,"""1/10/2011 10:04""",-11062.06,12346.0,"""Australia"""
"""25%""",null,null,null,1.0,null,1.25,13953.0,null
"""50%""",null,null,null,3.0,null,2.08,15152.0,null
"""75%""",null,null,null,10.0,null,4.13,16791.0,null
"""max""","""C581569""","""m""","""wrongly sold sets""",80995.0,"""9/9/2011 9:52""",38970.0,18287.0,"""Unspecified"""


In [5]:
null_summary = (
    raw.null_count()
    .transpose(include_header=True, header_name='column', column_names=['null_count'])
    .with_columns((pl.col('null_count') / raw.height * 100).round(2).alias('null_%'))
)
display(null_summary)

n_dup_rows = raw.height - raw.unique().height
print(f'Fully duplicated rows (extra copies) : {n_dup_rows:,} ({n_dup_rows / raw.height:.2%})')

display(
    raw.select(pl.all().n_unique())
    .transpose(include_header=True, header_name='column', column_names=['nunique'])
)

column,null_count,null_%
str,u32,f64
"""InvoiceNo""",0,0.0
"""StockCode""",0,0.0
"""Description""",1454,0.27
"""Quantity""",0,0.0
"""InvoiceDate""",0,0.0
"""UnitPrice""",0,0.0
"""CustomerID""",135080,24.93
"""Country""",0,0.0


Fully duplicated rows (extra copies) : 5,268 (0.97%)


column,nunique
str,u32
"""InvoiceNo""",25900
"""StockCode""",4070
"""Description""",4224
"""Quantity""",722
"""InvoiceDate""",23260
"""UnitPrice""",1630
"""CustomerID""",4373
"""Country""",38


In [6]:
print(f"Cancellation invoices (start with 'C') : {raw.filter(pl.col('InvoiceNo').str.starts_with('C')).height:,}")
print(f"Rows with Quantity  <= 0               : {raw.filter(pl.col('Quantity') <= 0).height:,}")
print(f"Rows with UnitPrice <= 0               : {raw.filter(pl.col('UnitPrice') <= 0).height:,}")
print(f"Rows without CustomerID                : {raw.filter(pl.col('CustomerID').is_null()).height:,}")

Cancellation invoices (start with 'C') : 9,288
Rows with Quantity  <= 0               : 10,624
Rows with UnitPrice <= 0               : 2,517
Rows without CustomerID                : 135,080


**What the overview shows**

- The file has 25,900 invoices, 4,070 stock codes and 38 countries. `CustomerID` is missing in **24.9%** of rows, so customer-level work can only use identified customers; revenue-level work keeps all rows.
- Extreme values exist: `Quantity` runs from -80,995 to 80,995 and `UnitPrice` goes as low as -11,062. These are not normal sales lines, so each one is checked before it is dropped.
- Four things need a decision: **5,268** exact duplicates (0.97%), **9,288** cancellation lines (`InvoiceNo` starts with `C`), **2,517** zero/negative-price lines and **1,454** missing descriptions (0.27%).

### 1.2 Cleaning

**Step 1 — snake_case columns, parse dates, drop exact duplicates.**

In [7]:
rename_map = {
    'InvoiceNo': 'invoice_no', 'StockCode': 'stock_code', 'Description': 'description',
    'Quantity': 'quantity', 'InvoiceDate': 'invoice_date', 'UnitPrice': 'unit_price',
    'CustomerID': 'customer_id', 'Country': 'country'
}

df = raw.rename(rename_map).with_columns(
    pl.col('invoice_date').str.to_datetime('%m/%d/%Y %H:%M')
)

rows_before = df.height
df = df.unique(maintain_order=True)
print(f'Before dedup       : {rows_before:,}')
print(f'After dedup        : {df.height:,}')
print(f'Reduced row count  : {rows_before - df.height:,} ({(rows_before - df.height) / rows_before:.2%})')
print(f"Period             : {df['invoice_date'].min()}  ->  {df['invoice_date'].max()}")

Before dedup       : 541,909
After dedup        : 536,641
Reduced row count  : 5,268 (0.97%)
Period             : 2010-12-01 08:26:00  ->  2011-12-09 12:50:00


> Removed **5,268** exact duplicate lines (0.97%). Assumption: identical lines are scan or export duplicates, not genuine double purchases; the effect is below 1% of rows.

**Step 2 — remove non-product `stock_code` rows** (postage, fees, manual adjustments, bad-debt entries, gift vouchers). They are not merchandise sales, so they would distort product and revenue analysis.

In [8]:
NON_PRODUCT_CODES = ['POST', 'DOT', 'M', 'm', 'C2', 'D', 'S', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'B', 'PADS']
is_non_product = pl.col('stock_code').is_in(NON_PRODUCT_CODES) | pl.col('stock_code').str.starts_with('gift_')

non_product_summary = (
    df.filter(is_non_product)
    .group_by('stock_code')
    .agg(
        rows=pl.len(),
        net_value=(pl.col('quantity') * pl.col('unit_price')).sum().round(2),
        description=pl.col('description').first()
    )
    .sort('rows', descending=True)
)
display(non_product_summary)

rows_before = df.height
df = df.filter(~is_non_product)
print(f'Before : {rows_before:,}')
print(f'After  : {df.height:,}')
print(f'Removed: {rows_before - df.height:,} ({(rows_before - df.height) / rows_before:.2%})')

stock_code,rows,net_value,description
str,u32,f64,str
"""POST""",1256,66230.64,"""POSTAGE"""
"""DOT""",710,206245.48,"""DOTCOM POSTAGE"""
"""M""",566,-69034.19,"""Manual"""
"""C2""",144,6986.0,"""CARRIAGE"""
"""D""",77,-5696.22,"""Discount"""
"""S""",62,-3039.65,"""SAMPLES"""
"""BANK CHARGES""",37,-7175.64,"""Bank Charges"""
"""AMAZONFEE""",34,-221520.5,"""AMAZON FEE"""
"""CRUK""",16,-7933.43,"""CRUK Commission"""


Before : 536,641
After  : 533,697
Removed: 2,944 (0.55%)


> Removed **2,944** non-product rows (0.55%): postage (`POST`, `DOT`), manual adjustments (`M`), discounts, fees (`AMAZONFEE` alone is -221,521), `BANK CHARGES`, `CRUK`, `B` (bad debt) and gift vouchers. They are costs or accounting entries, not merchandise sales. `DCGS...` codes are real products and stay.

**Step 3 — remove rows with `unit_price <= 0`.** Check what these rows are before dropping them.

In [9]:
zero_price = df.filter(pl.col('unit_price') <= 0)
print(f'Rows with unit_price <= 0          : {zero_price.height:,}')
print(f"  without customer_id              : {zero_price['customer_id'].is_null().sum():,} ({zero_price['customer_id'].is_null().mean():.1%})")
print(f"  without description              : {zero_price['description'].is_null().sum():,} ({zero_price['description'].is_null().mean():.1%})")
print(f"  with quantity <= 0 (non-cancel)  : {zero_price.filter(~pl.col('invoice_no').str.starts_with('C') & (pl.col('quantity') <= 0)).height:,}")

nonzero_missing_desc = df.filter((pl.col('unit_price') > 0) & pl.col('description').is_null()).height
print(f'Missing description on priced rows : {nonzero_missing_desc:,}')

rows_before = df.height
df = df.filter(pl.col('unit_price') > 0)
print(f'\nBefore : {rows_before:,}')
print(f'After  : {df.height:,}')
print(f'Removed: {rows_before - df.height:,} ({(rows_before - df.height) / rows_before:.2%})')

Rows with unit_price <= 0          : 2,492
  without customer_id              : 2,459 (98.7%)
  without description              : 1,446 (58.0%)
  with quantity <= 0 (non-cancel)  : 1,336
Missing description on priced rows : 0

Before : 533,697
After  : 531,205
Removed: 2,492 (0.47%)


> Removed **2,492** rows with `unit_price <= 0` (0.47%). Of these, 98.7% have no `customer_id`, 58.0% have no description, and 1,336 are non-cancellation lines with `quantity <= 0`, which points to stock write-offs and adjustments rather than sales. After this step no priced line is missing a description, and no non-cancellation line has `quantity <= 0`.

**Step 4 — split sales and returns, add derived columns.** Invoices starting with `C` are cancellations (returns). They stay in the analysis as negative revenue, so that `net revenue = sales + returns`.

In [10]:
df = df.with_columns(
    is_cancel=pl.col('invoice_no').str.starts_with('C'),
    revenue=pl.col('quantity') * pl.col('unit_price'),
    month=pl.col('invoice_date').dt.truncate('1mo'),
    weekday=pl.col('invoice_date').dt.weekday(),
    hour=pl.col('invoice_date').dt.hour(),
)

sales = df.filter(~pl.col('is_cancel'))
returns = df.filter(pl.col('is_cancel'))

print(f'Clean lines    : {df.height:,}')
print(f'Sales lines    : {sales.height:,}')
print(f'Return lines   : {returns.height:,}')

# Verification of derived columns
print('\nVerification')
print(f"  sales   quantity min / max   : {sales['quantity'].min()} / {sales['quantity'].max():,}")
print(f"  returns quantity min / max   : {returns['quantity'].min():,} / {returns['quantity'].max()}")
print(f"  sales rows with revenue <= 0 : {sales.filter(pl.col('revenue') <= 0).height}")
print(f"  returns rows with revenue >= 0: {returns.filter(pl.col('revenue') >= 0).height}")
print(f"  null descriptions            : {df['description'].null_count()}")
print(f"  hour range                   : {df['hour'].min()} - {df['hour'].max()}")
print(f"  weekdays present (1=Mon)     : {sorted(df['weekday'].unique().to_list())}")

Clean lines    : 531,205
Sales lines    : 522,537
Return lines   : 8,668

Verification
  sales   quantity min / max   : 1 / 80,995
  returns quantity min / max   : -80,995 / -1
  sales rows with revenue <= 0 : 0
  returns rows with revenue >= 0: 0
  null descriptions            : 0
  hour range                   : 6 - 20
  weekdays present (1=Mon)     : [1, 2, 3, 4, 5, 7]


> **522,537** sales lines and **8,668** return lines remain (531,205 clean lines in total). Verification passed: all sales quantities are positive, all return quantities negative, no null descriptions. Trading hours are 06:00-20:59 and there are **no Saturday transactions** (a feature of the business, not missing data; see 2.5).

**Step 5 — check the two structural features that affect interpretation: missing `customer_id` and extreme quantities.**

In [11]:
anon = df.filter(pl.col('customer_id').is_null())
print(f'Lines without customer_id : {anon.height:,} ({anon.height / df.height:.1%} of clean lines)')
print(f"Net revenue without id    : {anon['revenue'].sum():,.0f} ({anon['revenue'].sum() / df['revenue'].sum():.1%} of net revenue)")

multi_country = (
    df.filter(pl.col('customer_id').is_not_null())
    .group_by('customer_id').agg(n_countries=pl.col('country').n_unique())
    .filter(pl.col('n_countries') > 1)
)
print(f'Customers with >1 country : {multi_country.height}')

REVERSAL_QTY = 50_000
display(
    df.filter(pl.col('quantity').abs() >= REVERSAL_QTY)
    .select('invoice_no', 'invoice_date', 'stock_code', 'description', 'quantity', 'unit_price', 'customer_id', 'revenue')
    .sort('invoice_date')
)

# Each pair (sale + cancellation within minutes) sums to zero, so net revenue is unaffected.
# Order and customer counts would be inflated, so both sides of the pair are excluded from them.
sales_ex = sales.filter(pl.col('quantity') < REVERSAL_QTY)
returns_ex = returns.filter(pl.col('quantity') > -REVERSAL_QTY)
print(f'Net revenue effect of the pairs : {df.filter(pl.col("quantity").abs() >= REVERSAL_QTY)["revenue"].sum():,.2f}')
print(f'Sales lines  : {sales.height:,} -> {sales_ex.height:,}')
print(f'Return lines : {returns.height:,} -> {returns_ex.height:,}')

Lines without customer_id : 131,549 (24.8% of clean lines)
Net revenue without id    : 1,505,841 (15.4% of net revenue)
Customers with >1 country : 8


invoice_no,invoice_date,stock_code,description,quantity,unit_price,customer_id,revenue
str,datetime[μs],str,str,i64,f64,i64,f64
"""541431""",2011-01-18 10:01:00,"""23166""","""MEDIUM CERAMIC TOP STORAGE JAR""",74215,1.04,12346,77183.6
"""C541433""",2011-01-18 10:17:00,"""23166""","""MEDIUM CERAMIC TOP STORAGE JAR""",-74215,1.04,12346,-77183.6
"""581483""",2011-12-09 09:15:00,"""23843""","""PAPER CRAFT , LITTLE BIRDIE""",80995,2.08,16446,168469.6
"""C581484""",2011-12-09 09:27:00,"""23843""","""PAPER CRAFT , LITTLE BIRDIE""",-80995,2.08,16446,-168469.6


Net revenue effect of the pairs : 0.00
Sales lines  : 522,537 -> 522,535
Return lines : 8,668 -> 8,666


**Decisions**

- **Missing `customer_id`:** 131,549 clean lines (24.8%) carry 1,505,841 of net revenue (15.4%). Keep them for revenue analysis, exclude them from customer-level analysis. They hold a smaller share of revenue than of lines, so anonymous baskets are smaller than identified ones.
- **Customers in more than one country:** 8. Each is assigned to the country where they ordered most often.
- **Two fully reversed orders:** customer `12346` (74,215 units, 77,184, cancelled after 16 minutes) and customer `16446` (80,995 units, 168,470, cancelled after 12 minutes). Each pair sums to zero, so net revenue is unaffected, but together they would add **245,653** to gross sales and to returns and distort order and customer counts. Both sides of each pair are excluded from order and customer metrics (`sales_ex`, `returns_ex`).

**Step 6 — partial-period guard.** Check how complete each month is before comparing months.

In [12]:
display(
    df.group_by('month')
    .agg(first_day=pl.col('invoice_date').min().dt.date(), last_day=pl.col('invoice_date').max().dt.date(), lines=pl.len())
    .sort('month')
)

month,first_day,last_day,lines
datetime[μs],date,date,u32
2010-12-01 00:00:00,2010-12-01,2010-12-23,41515
2011-01-01 00:00:00,2011-01-04,2011-01-31,34580
2011-02-01 00:00:00,2011-02-01,2011-02-28,27155
2011-03-01 00:00:00,2011-03-01,2011-03-31,35940
2011-04-01 00:00:00,2011-04-01,2011-04-28,29272
2011-05-01 00:00:00,2011-05-01,2011-05-31,36290
2011-06-01 00:00:00,2011-06-01,2011-06-30,36179
2011-07-01 00:00:00,2011-07-01,2011-07-31,38834
2011-08-01 00:00:00,2011-08-01,2011-08-31,34723


> Data ends on **9 Dec 2011**, so Dec 2011 has 9 days (25,100 lines) and is partial. It is excluded from month-over-month comparisons and from the cohort analysis; where it is shown it is marked grey. Dec 2010 ends on 23 Dec (no later transactions in the file) and is treated as complete.

## 2. Revenue Analysis

### 2.1 Headline KPIs

**Question:** How much revenue does the business make, and how much of it comes back as returns?

In [13]:
gross_sales = sales_ex['revenue'].sum()
returns_value = returns_ex['revenue'].sum()          # negative number
net_revenue = gross_sales + returns_value
assert abs(net_revenue - df['revenue'].sum()) < 1    # same as net revenue of all clean lines

order_values = sales_ex.group_by('invoice_no').agg(order_value=pl.col('revenue').sum())
n_orders = order_values.height
n_customers = sales_ex['customer_id'].drop_nulls().n_unique()

return_rate = -returns_value / gross_sales * 100
return_rate_raw = -returns['revenue'].sum() / sales['revenue'].sum() * 100

print(f'Gross sales revenue          : {gross_sales:>12,.0f}')
print(f'Returns (cancellations)      : {returns_value:>12,.0f}')
print(f'Net revenue                  : {net_revenue:>12,.0f}')
print(f'Return rate (value)          : {return_rate:>11.1f}%   (raw, with the 2 reversed orders: {return_rate_raw:.1f}%)')
print(f'Orders (invoices)            : {n_orders:>12,}')
print(f'Identified customers         : {n_customers:>12,}')
print(f"Order value  mean / median   : {order_values['order_value'].mean():,.0f} / {order_values['order_value'].median():,.0f}")
print(f"Mean / median skew ratio     : {order_values['order_value'].mean() / order_values['order_value'].median():.2f}x")

Gross sales revenue          :   10,001,566
Returns (cancellations)      :     -230,248
Net revenue                  :    9,771,318
Return rate (value)          :         2.3%   (raw, with the 2 reversed orders: 4.6%)
Orders (invoices)            :       19,771
Identified customers         :        4,333
Order value  mean / median   : 506 / 302
Mean / median skew ratio     : 1.67x


> Net revenue is **9,771,318**: 10,001,566 gross sales minus 230,248 returns. Returns are **2.3%** of gross value; the raw figure of 4.6% is doubled by the two reversed orders, so 2.3% is the one to use. The median order is **302** vs a mean of **506** (1.67x), so a few very large orders pull the mean up and the median describes the typical order.

### 2.2 Monthly trend

**Question:** When does revenue peak, and is growth driven by more orders or bigger orders?

In [14]:
PARTIAL_MONTH = datetime(2011, 12, 1)

monthly = (
    df.group_by('month').agg(net_revenue=pl.col('revenue').sum())
    .join(sales_ex.group_by('month').agg(orders=pl.col('invoice_no').n_unique()), on='month')
    .sort('month')
    .with_columns(
        month_label=pl.col('month').dt.strftime('%Y-%m'),
        period=pl.when(pl.col('month') == PARTIAL_MONTH).then(pl.lit('Partial')).otherwise(pl.lit('Complete')),
        mom_change_pct=(pl.col('net_revenue') / pl.col('net_revenue').shift(1) - 1) * 100,
    )
    .with_columns(net_per_order=pl.col('net_revenue') / pl.col('orders'))
)
display(monthly.select('month_label', 'period', 'net_revenue', 'orders', 'net_per_order', 'mom_change_pct').with_columns(pl.col(pl.Float64).round(1)))

complete = monthly.filter(pl.col('period') == 'Complete')
nov = complete.filter(pl.col('month_label') == '2011-11')
jan_aug = complete.filter(pl.col('month_label').is_between(pl.lit('2011-01'), pl.lit('2011-08')))
sep_nov = complete.filter(pl.col('month_label').is_between(pl.lit('2011-09'), pl.lit('2011-11')))

nov_rev = nov['net_revenue'][0]
jan_aug_avg = jan_aug['net_revenue'].mean()
sep_nov_share = sep_nov['net_revenue'].sum() / complete['net_revenue'].sum() * 100
orders_growth = nov['orders'][0] / jan_aug['orders'].mean()
npo_growth = nov['net_per_order'][0] / jan_aug['net_per_order'].mean()

fig_monthly = make_subplots(specs=[[{'secondary_y': True}]])
fig_monthly.add_trace(
    go.Bar(
        x=monthly['month_label'].to_list(), y=monthly['net_revenue'].to_list(), name='Net revenue',
        marker_color=[BLUE if p == 'Complete' else '#BBBBBB' for p in monthly['period']]
    ),
    secondary_y=False
)
fig_monthly.add_trace(
    go.Scatter(x=monthly['month_label'].to_list(), y=monthly['orders'].to_list(), name='Orders', mode='lines+markers', line_color=ORANGE),
    secondary_y=True
)
fig_monthly.update_layout(title='Monthly net revenue and order count (grey bar = partial month, data ends 9 Dec 2011)', template='plotly_white')
fig_monthly.update_xaxes(title_text='Month')
fig_monthly.update_yaxes(title_text='Net revenue (GBP)', tickformat=',', secondary_y=False)
fig_monthly.update_yaxes(title_text='Orders (invoices)', secondary_y=True)
fig_monthly.show()

# Like-for-like check for the partial month: same 9-day window (1-9 Dec) in both years
dec_window = {
    year: df.filter((pl.col('invoice_date') >= datetime(year, 12, 1)) & (pl.col('invoice_date') < datetime(year, 12, 10)))['revenue'].sum()
    for year in (2010, 2011)
}
print(f"1-9 Dec net revenue: 2010 = {dec_window[2010]:,.0f} | 2011 = {dec_window[2011]:,.0f} | change = {dec_window[2011] / dec_window[2010] - 1:+.1%}")

print(f'Takeaway: Nov 2011 net revenue is {nov_rev / jan_aug_avg:.2f}x the Jan-Aug 2011 monthly average; '
      f'Sep-Nov hold {sep_nov_share:.1f}% of the 12 complete months.')
print(f'Takeaway: vs the Jan-Aug average, Nov has {orders_growth:.2f}x the orders but {npo_growth:.2f}x the net revenue per order -> growth comes from order volume.')

month_label,period,net_revenue,orders,net_per_order,mom_change_pct
str,str,f64,u32,f64,f64
"""2010-12""","""Complete""",758090.5,1550,489.1,null
"""2011-01""","""Complete""",578855.2,1080,536.0,-23.6
"""2011-02""","""Complete""",499464.4,1093,457.0,-13.7
"""2011-03""","""Complete""",679329.2,1440,471.8,36.0
"""2011-04""","""Complete""",482061.9,1235,390.3,-29.0
"""2011-05""","""Complete""",731004.8,1668,438.3,51.6
"""2011-06""","""Complete""",723845.1,1525,474.7,-1.0
"""2011-07""","""Complete""",676812.1,1452,466.1,-6.5
"""2011-08""","""Complete""",701289.9,1339,523.7,3.6


1-9 Dec net revenue: 2010 = 401,142 | 2011 = 440,400 | change = +9.8%
Takeaway: Nov 2011 net revenue is 2.25x the Jan-Aug 2011 monthly average; Sep-Nov hold 37.5% of the 12 complete months.
Takeaway: vs the Jan-Aug average, Nov has 2.03x the orders but 1.10x the net revenue per order -> growth comes from order volume.


**Result:** Net revenue is flat to slowly rising from Dec 2010 to Aug 2011 (482k-758k a month), then climbs: Sep 1.01M, Oct 1.06M and **Nov 1.43M**, which is **2.25x** the Jan-Aug average. Sep-Nov hold **37.5%** of the 12 complete months.

The peak is a volume effect: Nov has 2.03x the orders of an average Jan-Aug month but only 1.10x the net revenue per order. Dec 2011 (440k) covers 9 days only; on the same 1-9 Dec window it is **+9.8%** vs Dec 2010 (401k), an indicative like-for-like comparison, based on one year of history.

### 2.3 Revenue by country

**Question:** How dependent is revenue on the home market, and where do foreign customers look different?

In [15]:
country_summary = (
    df.group_by('country').agg(net_revenue=pl.col('revenue').sum())
    .join(
        df.filter(pl.col('customer_id').is_not_null())
        .group_by('country').agg(customers=pl.col('customer_id').n_unique(), identified_revenue=pl.col('revenue').sum()),
        on='country', how='left'
    )
    .join(sales_ex.group_by('country').agg(orders=pl.col('invoice_no').n_unique()), on='country', how='left')
    .with_columns(
        pl.col('customers').fill_null(0),
        **{'revenue_share_%': pl.col('net_revenue') / pl.col('net_revenue').sum() * 100}
    )
    .with_columns(revenue_per_customer=pl.when(pl.col('customers') > 0).then(pl.col('identified_revenue') / pl.col('customers')))
    .sort('net_revenue', descending=True)
)
display(country_summary.head(10).with_columns(pl.col(pl.Float64).round(1)))

uk_share = country_summary.filter(pl.col('country') == 'United Kingdom')['revenue_share_%'][0]
uk_rpc = country_summary.filter(pl.col('country') == 'United Kingdom')['revenue_per_customer'][0]
non_uk_top = country_summary.filter(pl.col('country') != 'United Kingdom').head(10)

fig_country = px.bar(
    non_uk_top, x='net_revenue', y='country', orientation='h', color='customers',
    color_continuous_scale='Blues', text_auto=',.0f', template='plotly_white',
    title='Top 10 non-UK countries by net revenue (colour = identified customers)',
    labels={'net_revenue': 'Net revenue (GBP)', 'country': 'Country', 'customers': 'Customers'}
)
fig_country.update_yaxes(categoryorder='total ascending')
fig_country.show()

nl = country_summary.filter(pl.col('country') == 'Netherlands')
print(f'Takeaway: the UK brings {uk_share:.1f}% of net revenue; the top 10 foreign countries together bring {non_uk_top["revenue_share_%"].sum():.1f}%.')
print(f"Takeaway: Netherlands has {nl['customers'][0]} customers but {nl['revenue_per_customer'][0]:,.0f} per customer vs {uk_rpc:,.0f} in the UK ({nl['revenue_per_customer'][0] / uk_rpc:.0f}x).")

country,net_revenue,customers,identified_revenue,orders,revenue_share_%,revenue_per_customer
str,f64,u32,f64,u32,f64,f64
"""United Kingdom""",8281390.4,3942,6802545.6,17899,84.8,1725.7
"""Netherlands""",283479.5,9,283479.5,93,2.9,31497.7
"""EIRE""",259380.0,3,246732.5,282,2.7,82244.2
"""Germany""",200619.7,95,200619.7,443,2.1,2111.8
"""France""",182076.6,87,181385.5,382,1.9,2084.9
"""Australia""",136922.5,9,136922.5,56,1.4,15213.6
"""Switzerland""",52483.0,21,51859.4,50,0.5,2469.5
"""Spain""",51746.6,30,51746.6,88,0.5,1724.9
"""Belgium""",36663.0,25,36663.0,98,0.4,1466.5


Takeaway: the UK brings 84.8% of net revenue; the top 10 foreign countries together bring 13.0%.
Takeaway: Netherlands has 9 customers but 31,498 per customer vs 1,726 in the UK (18x).


**Result:** The UK brings **84.8%** of net revenue and the top 10 foreign countries together only 13.0%. Revenue per customer is very uneven abroad: Netherlands (9 customers) 31,498, EIRE (3 customers) 82,244 and Australia (9 customers) 15,214, against 1,726 in the UK. Germany (95 customers, 2,112) and France (87 customers, 2,085) look like the UK.

The Netherlands, EIRE and Australia revenue therefore rests on a handful of large accounts, while Germany and France are broad, UK-like markets. Revenue per customer uses identified revenue only; countries with fewer than 10 customers should not be ranked on per-customer values.

### 2.4 Products

**Question:** Is revenue spread across the catalogue or concentrated in a few items?

In [16]:
product_summary = (
    df.group_by('stock_code')
    .agg(
        description=pl.col('description').mode().sort().first(),
        net_revenue=pl.col('revenue').sum(),
        net_units=pl.col('quantity').sum()
    )
    .sort('net_revenue', descending=True)
    .with_columns(
        product_rank=pl.int_range(1, pl.len() + 1),
        **{'cum_revenue_share_%': pl.col('net_revenue').cum_sum() / pl.col('net_revenue').sum() * 100}
    )
    .with_columns(**{'catalogue_share_%': pl.col('product_rank') / pl.len() * 100})
)

n_products = product_summary.height
n_for_80 = product_summary.filter(pl.col('cum_revenue_share_%') < 80).height + 1
top10_share = product_summary.head(10)['net_revenue'].sum() / product_summary['net_revenue'].sum() * 100
negative_products = product_summary.filter(pl.col('net_revenue') < 0).height

display(product_summary.head(10).with_columns(pl.col(pl.Float64).round(1)))
print(f'Products sold                    : {n_products:,}')
print(f'Products with negative net value : {negative_products}')

fig_products = px.bar(
    product_summary.head(10).with_columns(label=pl.col('description').str.slice(0, 30)),
    x='net_revenue', y='label', orientation='h', text_auto=',.0f', template='plotly_white',
    color_discrete_sequence=[BLUE], title='Top 10 products by net revenue',
    labels={'net_revenue': 'Net revenue (GBP)', 'label': 'Product'}
)
fig_products.update_yaxes(categoryorder='total ascending')
fig_products.show()

fig_pareto = px.line(
    product_summary, x='catalogue_share_%', y='cum_revenue_share_%', template='plotly_white',
    title='Cumulative net revenue share by product rank (Pareto curve)',
    labels={'catalogue_share_%': 'Share of catalogue, ranked by revenue (%)', 'cum_revenue_share_%': 'Cumulative share of net revenue (%)'}
)
fig_pareto.update_traces(line_color=BLUE)
fig_pareto.add_hline(y=80, line_dash='dash', line_color=ORANGE)
fig_pareto.show()

print(f'Takeaway: the top 10 products make {top10_share:.1f}% of net revenue; {n_for_80:,} of {n_products:,} products ({n_for_80 / n_products:.1%}) make 80%.')

stock_code,description,net_revenue,net_units,product_rank,cum_revenue_share_%,catalogue_share_%
str,str,f64,i64,i64,f64,f64
"""22423""","""REGENCY CAKESTAND 3 TIER""",164459.5,12996,1,1.7,0.0
"""47566""","""PARTY BUNTING""",98243.9,18006,2,2.7,0.1
"""85123A""","""WHITE HANGING HEART T-LIGHT HOLDER""",97838.4,35063,3,3.7,0.1
"""85099B""","""JUMBO BAG RED RETROSPOT""",92175.8,47256,4,4.6,0.1
"""23084""","""RABBIT NIGHT LIGHT""",66661.6,30631,5,5.3,0.1
"""22086""","""PAPER CHAIN KIT 50'S CHRISTMAS """,63715.2,18876,6,6.0,0.2
"""84879""","""ASSORTED COLOUR BIRD ORNAMENT""",58792.4,36282,7,6.6,0.2
"""79321""","""CHILLI LIGHTS""",53746.7,10222,8,7.1,0.2
"""22502""","""PICNIC BASKET WICKER SMALL""",51023.5,1862,9,7.6,0.2


Products sold                    : 3,921
Products with negative net value : 19


Takeaway: the top 10 products make 8.2% of net revenue; 840 of 3,921 products (21.4%) make 80%.


**Result:** Revenue is spread across the catalogue. The top 10 products make only **8.2%** of net revenue (the leader, `REGENCY CAKESTAND 3 TIER`, makes 1.7%), and 840 of 3,921 products (**21.4%**) make 80%, close to a classic 80/20 Pareto. 19 products have negative net value (more returned than sold within the period, probably sold before Dec 2010). A single product failing would not move total revenue much.

### 2.5 When customers buy

**Question:** Which weekdays and hours carry the revenue?

In [17]:
WEEKDAYS = {1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat', 7: 'Sun'}

weekday_summary = (
    df.group_by('weekday').agg(net_revenue=pl.col('revenue').sum())
    .join(sales_ex.group_by('weekday').agg(orders=pl.col('invoice_no').n_unique()), on='weekday')
    .sort('weekday')
    .with_columns(
        day=pl.col('weekday').replace_strict(WEEKDAYS, return_dtype=pl.String),
        **{'revenue_share_%': pl.col('net_revenue') / pl.col('net_revenue').sum() * 100}
    )
)
hour_summary = (
    df.group_by('hour').agg(net_revenue=pl.col('revenue').sum())
    .sort('hour')
    .with_columns(**{'revenue_share_%': pl.col('net_revenue') / pl.col('net_revenue').sum() * 100})
)
missing_days = [WEEKDAYS[d] for d in WEEKDAYS if d not in weekday_summary['weekday'].to_list()]
print(f'Weekdays with no transactions : {missing_days}')
display(weekday_summary.select('day', 'net_revenue', 'orders', 'revenue_share_%').with_columns(pl.col(pl.Float64).round(1)))

fig_time = make_subplots(rows=1, cols=2, subplot_titles=('By weekday', 'By hour of day'))
fig_time.add_trace(go.Bar(x=weekday_summary['day'].to_list(), y=weekday_summary['net_revenue'].to_list(), marker_color=BLUE, name='Weekday'), row=1, col=1)
fig_time.add_trace(go.Bar(x=hour_summary['hour'].to_list(), y=hour_summary['net_revenue'].to_list(), marker_color=ORANGE, name='Hour'), row=1, col=2)
fig_time.update_layout(title='Net revenue by weekday and by hour of day', template='plotly_white', showlegend=False)
fig_time.update_xaxes(title_text='Weekday', row=1, col=1)
fig_time.update_xaxes(title_text='Hour of day', row=1, col=2)
fig_time.update_yaxes(title_text='Net revenue (GBP)', tickformat=',', row=1, col=1)
fig_time.update_yaxes(title_text='Net revenue (GBP)', tickformat=',', row=1, col=2)
fig_time.show()

peak_hours_share = hour_summary.filter(pl.col('hour').is_between(10, 15))['revenue_share_%'].sum()
thu_share = weekday_summary.filter(pl.col('day') == 'Thu')['revenue_share_%'][0]
sun_share = weekday_summary.filter(pl.col('day') == 'Sun')['revenue_share_%'][0]
print(f'Takeaway: 10:00-15:59 holds {peak_hours_share:.1f}% of net revenue; Thursday {thu_share:.1f}% vs Sunday {sun_share:.1f}%.')

Weekdays with no transactions : ['Sat']


day,net_revenue,orders,revenue_share_%
str,f64,u32,f64
"""Mon""",1622206.5,3076,16.6
"""Tue""",1968073.6,3508,20.1
"""Wed""",1738840.8,3667,17.8
"""Thu""",2080487.2,4209,21.3
"""Fri""",1570910.1,3108,16.1
"""Sun""",790799.9,2203,8.1


Takeaway: 10:00-15:59 holds 75.6% of net revenue; Thursday 21.3% vs Sunday 8.1%.


**Result:** Trade is a weekday-daytime pattern: **10:00-15:59 holds 75.6%** of net revenue, Thursday is the strongest day (**21.3%**), Sunday is the weakest (**8.1%**), and there are no Saturday transactions. Evening trade after 18:00 is marginal. This is consistent with a business-customer base, but the data has no customer-type field to confirm it.

### 2.6 Returns

**Question:** Where do returns come from, and are they stable over time?

In [18]:
returns_monthly = (
    sales_ex.group_by('month').agg(gross_sales=pl.col('revenue').sum())
    .join(returns_ex.group_by('month').agg(returns_value=pl.col('revenue').sum()), on='month')
    .sort('month')
    .with_columns(
        month_label=pl.col('month').dt.strftime('%Y-%m'),
        **{'return_rate_%': -pl.col('returns_value') / pl.col('gross_sales') * 100}
    )
)

fig_returns = px.line(
    returns_monthly, x='month_label', y='return_rate_%', markers=True, template='plotly_white',
    title='Monthly return rate by value, excluding the two fully reversed orders (Dec 2011 is partial)',
    labels={'month_label': 'Month', 'return_rate_%': 'Returns as % of gross sales'}
)
fig_returns.update_traces(line_color=ORANGE)
fig_returns.show()

top_returned = (
    returns_ex.group_by('stock_code')
    .agg(description=pl.col('description').mode().sort().first(), returned_value=-pl.col('revenue').sum(), returned_units=-pl.col('quantity').sum())
    .join(sales_ex.group_by('stock_code').agg(gross_value=pl.col('revenue').sum()), on='stock_code', how='left')
    .with_columns(**{'return_rate_%': pl.col('returned_value') / pl.col('gross_value') * 100})
    .sort('returned_value', descending=True)
    .head(8)
)
display(top_returned.with_columns(pl.col(pl.Float64).round(1)))

max_row = returns_monthly.sort('return_rate_%', descending=True).row(0, named=True)
print(f"Takeaway: monthly return rate ranges {returns_monthly['return_rate_%'].min():.1f}%-{returns_monthly['return_rate_%'].max():.1f}% (peak {max_row['month_label']}); "
      f"the top 8 returned products carry {top_returned['returned_value'].sum() / -returns_ex['revenue'].sum():.1%} of returned value.")

stock_code,description,returned_value,returned_units,gross_value,return_rate_%
str,str,f64,i64,f64,f64
"""22423""","""REGENCY CAKESTAND 3 TIER""",9697.0,855,174156.5,5.6
"""85123A""","""WHITE HANGING HEART T-LIGHT HOLDER""",6624.3,2578,104462.8,6.3
"""21108""","""FAIRY CAKE FLANNEL ASSORTED COLOUR""",6591.4,3150,18056.3,36.5
"""23113""","""PANTRY CHOPPING BOARD""",4803.1,946,5868.5,81.8
"""48185""","""DOORMAT FAIRY CAKE""",4554.9,674,23872.9,19.1
"""21175""","""GIN + TONIC DIET METAL SIGN""",3775.3,2030,27030.0,14.0
"""47566B""","""TEA TIME PARTY BUNTING""",3692.9,1424,11275.3,32.8
"""22273""","""FELTCRAFT DOLL MOLLY""",3512.6,1447,12001.6,29.3


Takeaway: monthly return rate ranges 1.2%-6.5% (peak 2011-04); the top 8 returned products carry 18.8% of returned value.


**Result:** The monthly return rate (excluding the two reversed orders) ranges **1.2%-6.5%**, peaking in Apr 2011. The 8 largest returned products carry only 18.8% of returned value, so returns are spread rather than caused by a few products. Some products stand out on rate: `PANTRY CHOPPING BOARD` 81.8% (4,803 returned vs 5,869 sold), `FAIRY CAKE FLANNEL ASSORTED COLOUR` 36.5% and `TEA TIME PARTY BUNTING` 32.8%. The file has no return reason, and a cancellation can refer to a purchase made before the period, so product-level rates are a screening signal, not a verdict.

## 3. Customer-Level Analysis

Tests and segments compare customers, not invoice lines, so the customer table comes first. Only lines with a `customer_id` are used here.

### 3.1 Customer table

In [19]:
customer_sales = (
    sales_ex.filter(pl.col('customer_id').is_not_null())
    .group_by('customer_id')
    .agg(
        orders=pl.col('invoice_no').n_unique(),
        items=pl.col('quantity').sum(),
        products=pl.col('stock_code').n_unique(),
        gross_revenue=pl.col('revenue').sum(),
        first_order=pl.col('invoice_date').min(),
        last_order=pl.col('invoice_date').max(),
        country=pl.col('country').mode().sort().first()
    )
)
customer_returns = (
    returns_ex.filter(pl.col('customer_id').is_not_null())
    .group_by('customer_id').agg(returns_value=pl.col('revenue').sum())
)
print(f"Customers with returns but no sale in the period : {customer_returns.join(customer_sales, on='customer_id', how='anti').height}")

cust_statistic = (
    customer_sales.join(customer_returns, on='customer_id', how='left')
    .with_columns(pl.col('returns_value').fill_null(0.0))
    .with_columns(net_revenue=pl.col('gross_revenue') + pl.col('returns_value'))
    .with_columns(
        aov=pl.col('gross_revenue') / pl.col('orders'),
        is_uk=pl.col('country') == 'United Kingdom',
        is_repeat=pl.col('orders') >= 2
    )
)

rows_before = cust_statistic.height
cust_statistic = cust_statistic.filter(pl.col('net_revenue').round(2) > 0)
print(f'Before : {rows_before:,}')
print(f'After  : {cust_statistic.height:,}')
print(f'Removed: {rows_before - cust_statistic.height} customers with net revenue <= 0 (returns cancel out purchases)')

print('\nVerification')
print(f"  min orders / min net revenue : {cust_statistic['orders'].min()} / {cust_statistic['net_revenue'].min():,.2f}")
print(f"  duplicated customer_id       : {cust_statistic['customer_id'].is_duplicated().sum()}")
display(cust_statistic.head().with_columns(pl.col(pl.Float64).round(1)))

Customers with returns but no sale in the period : 28
Before : 4,333
After  : 4,320
Removed: 13 customers with net revenue <= 0 (returns cancel out purchases)

Verification
  min orders / min net revenue : 1 / 2.90
  duplicated customer_id       : 0


customer_id,orders,items,products,gross_revenue,first_order,last_order,country,returns_value,net_revenue,aov,is_uk,is_repeat
i64,u32,i64,u32,f64,datetime[μs],datetime[μs],str,f64,f64,f64,bool,bool
14300,1,32,3,43.2,2011-11-21 08:57:00,2011-11-21 08:57:00,"""United Kingdom""",0.0,43.2,43.2,true,false
17438,1,103,17,388.5,2011-03-31 10:26:00,2011-03-31 10:26:00,"""United Kingdom""",-10.2,378.3,388.5,true,false
13410,5,518,15,1003.1,2011-02-21 14:38:00,2011-11-15 15:33:00,"""United Kingdom""",-14.2,988.9,200.6,true,true
17462,2,227,25,675.7,2011-02-28 17:01:00,2011-10-10 13:56:00,"""United Kingdom""",-12.8,662.9,337.8,true,true
15211,2,240,52,894.4,2010-12-09 16:08:00,2011-10-10 13:31:00,"""United Kingdom""",0.0,894.4,447.2,true,true


> **4,333** identified customers have at least one sale; **13** (0.3%) are removed because returns cancel out their purchases (net revenue <= 0), leaving **4,320**. A further 28 customers appear only in returns (they bought before Dec 2010) and are not in the table. Verification passed: every customer has at least 1 order, minimum net revenue is 2.90, no duplicated `customer_id`.

### 3.2 Revenue concentration

**Question:** How much of net revenue comes from the best customers?

In [20]:
cust_ranked = (
    cust_statistic.sort('net_revenue', descending=True)
    .with_columns(
        **{'customer_rank_%': pl.int_range(1, pl.len() + 1) / pl.len() * 100,
           'cum_revenue_share_%': pl.col('net_revenue').cum_sum() / pl.col('net_revenue').sum() * 100}
    )
)
total_cust_revenue = cust_ranked['net_revenue'].sum()
top10_cust_share = cust_ranked.filter(pl.col('customer_rank_%') <= 10)['net_revenue'].sum() / total_cust_revenue * 100
top20_cust_share = cust_ranked.filter(pl.col('customer_rank_%') <= 20)['net_revenue'].sum() / total_cust_revenue * 100
bottom50_cust_share = cust_ranked.filter(pl.col('customer_rank_%') > 50)['net_revenue'].sum() / total_cust_revenue * 100
mean_rev, median_rev = cust_statistic['net_revenue'].mean(), cust_statistic['net_revenue'].median()

fig_conc = px.line(
    cust_ranked, x='customer_rank_%', y='cum_revenue_share_%', template='plotly_white',
    title='Cumulative net revenue share by customer rank (best customers first)',
    labels={'customer_rank_%': 'Share of customers, ranked by net revenue (%)', 'cum_revenue_share_%': 'Cumulative share of net revenue (%)'}
)
fig_conc.update_traces(line_color=BLUE)
fig_conc.add_shape(type='line', x0=0, y0=0, x1=100, y1=100, line=dict(dash='dash', color='#999999'))
fig_conc.show()

print(f'Mean vs median net revenue per customer : {mean_rev:,.0f} vs {median_rev:,.0f} ({mean_rev / median_rev:.1f}x -> right-skewed)')
print(f'Takeaway: the top 10% of customers bring {top10_cust_share:.1f}% of net revenue and the top 20% bring {top20_cust_share:.1f}%; the bottom 50% bring {bottom50_cust_share:.1f}%.')

Mean vs median net revenue per customer : 1,914 vs 650 (2.9x -> right-skewed)
Takeaway: the top 10% of customers bring 60.1% of net revenue and the top 20% bring 73.7%; the bottom 50% bring 8.1%.


**Result:** Net revenue per customer is strongly right-skewed: mean **1,914** vs median **650** (2.9x). The best **10%** of customers bring **60.1%** of net revenue, the best 20% bring 73.7%, and the bottom half of customers brings only 8.1%. Losing a handful of top accounts would hurt far more than losing many small ones.

### 3.3 Repeat behaviour

**Question:** How many customers come back, and how much revenue do repeat buyers carry?

In [21]:
order_bucket_labels = ['1 order', '2 orders', '3-4 orders', '5-9 orders', '10+ orders']

order_buckets = (
    cust_statistic.with_columns(
        order_bucket=pl.when(pl.col('orders') == 1).then(pl.lit('1 order'))
        .when(pl.col('orders') == 2).then(pl.lit('2 orders'))
        .when(pl.col('orders') <= 4).then(pl.lit('3-4 orders'))
        .when(pl.col('orders') <= 9).then(pl.lit('5-9 orders'))
        .otherwise(pl.lit('10+ orders'))
    )
    .group_by('order_bucket')
    .agg(customers=pl.len(), net_revenue=pl.col('net_revenue').sum())
    .with_columns(
        **{'customer_share_%': pl.col('customers') / pl.col('customers').sum() * 100,
           'revenue_share_%': pl.col('net_revenue') / pl.col('net_revenue').sum() * 100}
    )
)
order_buckets = order_buckets.with_columns(
    bucket_order=pl.col('order_bucket').replace_strict({b: i for i, b in enumerate(order_bucket_labels)}, return_dtype=pl.Int8)
).sort('bucket_order')
display(order_buckets.drop('bucket_order').with_columns(pl.col(pl.Float64).round(1)))

repeat_rate = cust_statistic['is_repeat'].mean() * 100
one_time = order_buckets.filter(pl.col('order_bucket') == '1 order')
heavy = order_buckets.filter(pl.col('order_bucket') == '10+ orders')

fig_orders = px.bar(
    order_buckets.unpivot(index='order_bucket', on=['customer_share_%', 'revenue_share_%'], variable_name='metric', value_name='share_%'),
    x='order_bucket', y='share_%', color='metric', barmode='group', text_auto='.1f', template='plotly_white',
    color_discrete_map={'customer_share_%': BLUE, 'revenue_share_%': ORANGE},
    category_orders={'order_bucket': order_bucket_labels},
    title='Share of customers vs share of net revenue by number of orders',
    labels={'order_bucket': 'Orders per customer', 'share_%': 'Share (%)', 'metric': 'Metric'}
)
fig_orders.show()

print(f'Repeat rate (2+ orders) : {repeat_rate:.1f}%')
print(f"Takeaway: one-time buyers are {one_time['customer_share_%'][0]:.1f}% of customers but {one_time['revenue_share_%'][0]:.1f}% of revenue; "
      f"customers with 10+ orders are {heavy['customer_share_%'][0]:.1f}% of customers but {heavy['revenue_share_%'][0]:.1f}% of revenue.")

order_bucket,customers,net_revenue,customer_share_%,revenue_share_%
str,u32,f64,f64,f64
"""1 order""",1494,536419.4,34.6,6.5
"""2 orders""",828,540174.5,19.2,6.5
"""3-4 orders""",897,1.152247e6,20.8,13.9
"""5-9 orders""",715,1696926.2,16.6,20.5
"""10+ orders""",386,4343870.2,8.9,52.5


Repeat rate (2+ orders) : 65.4%
Takeaway: one-time buyers are 34.6% of customers but 6.5% of revenue; customers with 10+ orders are 8.9% of customers but 52.5% of revenue.


**Result:** **65.4%** of customers ordered at least twice. One-time buyers are 34.6% of customers (1,494) but only **6.5%** of revenue, while the 386 customers with 10+ orders are 8.9% of customers but **52.5%** of revenue. Revenue depends on a small group of frequent buyers, and the first-to-second order step is where about a third of customers stop.

### 3.4 RFM segmentation

**Method:** snapshot = last invoice date + 1 day. Recency = days since last order, Frequency = number of orders, Monetary = net revenue. Each metric is scored 1–5 with `qcut` on the rank (`ordinal` rank breaks ties by row order, so identical values can land in neighbouring score bins — the usual trade-off when many customers share `orders = 1`). Segment labels are rule-based and written as a vectorised `when/then` chain.

In [22]:
snapshot = df['invoice_date'].max() + timedelta(days=1)

rfm = (
    cust_statistic.select('customer_id', 'last_order', 'orders', 'net_revenue')
    .with_columns(recency_days=(pl.lit(snapshot) - pl.col('last_order')).dt.total_days())
    .with_columns(
        r_score=pl.col('recency_days').rank('ordinal').qcut(5, labels=['5', '4', '3', '2', '1']).cast(pl.String).cast(pl.Int8),
        f_score=pl.col('orders').rank('ordinal').qcut(5, labels=['1', '2', '3', '4', '5']).cast(pl.String).cast(pl.Int8),
        m_score=pl.col('net_revenue').rank('ordinal').qcut(5, labels=['1', '2', '3', '4', '5']).cast(pl.String).cast(pl.Int8),
    )
    .with_columns(
        segment=pl.when((pl.col('r_score') >= 4) & (pl.col('f_score') >= 4)).then(pl.lit('Champions'))
        .when((pl.col('r_score') <= 2) & (pl.col('f_score') >= 4)).then(pl.lit("Can't Lose Them"))
        .when((pl.col('r_score') >= 3) & (pl.col('f_score') >= 3)).then(pl.lit('Loyal Customers'))
        .when((pl.col('r_score') >= 4) & (pl.col('f_score') == 1)).then(pl.lit('New Customers'))
        .when(pl.col('r_score') >= 4).then(pl.lit('Potential Loyalists'))
        .when((pl.col('r_score') <= 2) & (pl.col('f_score') >= 3)).then(pl.lit('At Risk'))
        .when(pl.col('r_score') == 3).then(pl.lit('Need Attention'))
        .otherwise(pl.lit('Hibernating'))
    )
)

print(f'Snapshot date : {snapshot}')
print(f"Recency days min / median / max : {rfm['recency_days'].min()} / {rfm['recency_days'].median():.0f} / {rfm['recency_days'].max()}")
print(f"Unassigned customers            : {rfm['segment'].null_count()}")

segment_summary = (
    rfm.group_by('segment')
    .agg(
        customers=pl.len(),
        net_revenue=pl.col('net_revenue').sum(),
        avg_recency_days=pl.col('recency_days').mean(),
        avg_orders=pl.col('orders').mean(),
        avg_net_revenue=pl.col('net_revenue').mean()
    )
    .with_columns(
        **{'customer_share_%': pl.col('customers') / pl.col('customers').sum() * 100,
           'revenue_share_%': pl.col('net_revenue') / pl.col('net_revenue').sum() * 100}
    )
    .sort('net_revenue', descending=True)
)
display(segment_summary.with_columns(pl.col(pl.Float64).round(1)))

fig_rfm = px.scatter(
    segment_summary, x='avg_recency_days', y='avg_orders', size='customers', color='revenue_share_%', text='segment',
    size_max=60, color_continuous_scale='Blues', template='plotly_white',
    title='RFM segments: recency vs frequency (bubble size = customers, colour = revenue share)',
    labels={'avg_recency_days': 'Average recency (days since last order, reversed axis)', 'avg_orders': 'Average orders per customer', 'revenue_share_%': 'Revenue share (%)'}
)
fig_rfm.update_xaxes(autorange='reversed')
fig_rfm.update_traces(textposition='top center')
fig_rfm.show()

top2 = segment_summary.filter(pl.col('segment').is_in(['Champions', "Can't Lose Them"]))
at_risk = segment_summary.filter(pl.col('segment').is_in(['At Risk', "Can't Lose Them"]))
print(f"Takeaway: Champions + Can't Lose Them = {top2['customer_share_%'].sum():.1f}% of customers and {top2['revenue_share_%'].sum():.1f}% of net revenue.")
print(f"Takeaway: At Risk + Can't Lose Them = {at_risk['customer_share_%'].sum():.1f}% of customers and {at_risk['revenue_share_%'].sum():.1f}% of net revenue are no longer recent.")

Snapshot date : 2011-12-10 12:50:00
Recency days min / median / max : 1 / 51 / 374
Unassigned customers            : 0


segment,customers,net_revenue,avg_recency_days,avg_orders,avg_net_revenue,customer_share_%,revenue_share_%
str,u32,f64,f64,f64,f64,f64,f64
"""Champions""",1109,5614253.9,12.9,10.1,5062.4,25.7,67.9
"""Loyal Customers""",823,1188557.3,37.1,3.7,1444.2,19.1,14.4
"""Can't Lose Them""",297,488529.4,133.6,4.8,1644.9,6.9,5.9
"""Hibernating""",1068,416433.6,217.7,1.1,389.9,24.7,5.0
"""At Risk""",363,296366.5,162.9,2.3,816.4,8.4,3.6
"""Need Attention""",347,150500.6,52.6,1.2,433.7,8.0,1.8
"""Potential Loyalists""",181,74553.4,16.3,1.4,411.9,4.2,0.9
"""New Customers""",132,40442.6,19.7,1.0,306.4,3.1,0.5


Takeaway: Champions + Can't Lose Them = 32.5% of customers and 73.8% of net revenue.
Takeaway: At Risk + Can't Lose Them = 15.3% of customers and 9.5% of net revenue are no longer recent.


**Result:** **Champions** are 25.8% of customers (1,115) and bring **67.9%** of net revenue, ordering 10.0 times on average with the last order 12.9 days before the snapshot. Together with `Can't Lose Them` they hold 73.8% of revenue.

The risk sits in the customers who used to buy often and went quiet: `Can't Lose Them` (291 customers, 4.8 orders, 130 days since last order) and `At Risk` (351 customers, 164 days) together are 14.9% of customers and **9.6% of net revenue (about 792k)**. `Hibernating` is the largest low-value group (25.1% of customers, 5.0% of revenue). `New Customers` and `Potential Loyalists` are small (3.2% and 3.8% of customers), so the pipeline of new buyers is thin at the snapshot date.

### 3.5 Cohort retention

**Method:** cohort = month of the customer's first order in the dataset; retention = share of the cohort that orders again in month *n* after the first order. December 2011 is partial, so both cohorts and activity stop at November 2011. Customers whose first order falls in Dec 2010 may have bought before the dataset starts, so that cohort mixes new and existing customers.

In [23]:
known_sales = sales_ex.filter(pl.col('customer_id').is_not_null())

first_month = known_sales.group_by('customer_id').agg(cohort_month=pl.col('month').min())

activity = (
    known_sales.filter(pl.col('month') < PARTIAL_MONTH)
    .select('customer_id', 'month').unique()
    .join(first_month, on='customer_id')
    .with_columns(
        cohort_index=(pl.col('month').dt.year() * 12 + pl.col('month').dt.month())
        - (pl.col('cohort_month').dt.year() * 12 + pl.col('cohort_month').dt.month())
    )
)

cohort_counts = activity.group_by('cohort_month', 'cohort_index').agg(active=pl.col('customer_id').n_unique())
cohort_size = cohort_counts.filter(pl.col('cohort_index') == 0).select('cohort_month', cohort_size=pl.col('active'))

retention = (
    cohort_counts.join(cohort_size, on='cohort_month')
    .with_columns(retention_rate=pl.col('active') / pl.col('cohort_size') * 100)
)
retention_pivot = retention.pivot(on='cohort_index', index='cohort_month', values='retention_rate').sort('cohort_month')
period_cols = sorted([c for c in retention_pivot.columns if c != 'cohort_month'], key=int)

print(f"Cohorts: {retention_pivot.height}   max period index: {max(int(c) for c in period_cols)}")
display(cohort_size.sort('cohort_month').with_columns(pl.col('cohort_month').dt.strftime('%Y-%m')))

fig_cohort = px.imshow(
    retention_pivot.select(period_cols).to_numpy(),
    x=period_cols, y=retention_pivot['cohort_month'].dt.strftime('%Y-%m').to_list(),
    color_continuous_scale='Blues', text_auto='.0f', aspect='auto', template='plotly_white',
    title='Monthly cohort retention (% of cohort active in month n after first order)',
    labels={'x': 'Months since first order', 'y': 'Cohort (first order month)', 'color': 'Retention %'}
)
fig_cohort.update_xaxes(side='top')
fig_cohort.show()

ret_by_period = retention.group_by('cohort_index').agg(avg_retention=pl.col('retention_rate').mean(), cohorts=pl.len()).sort('cohort_index')
def avg_ret(i):
    return ret_by_period.filter(pl.col('cohort_index') == i)['avg_retention'][0]
dec10 = retention.filter((pl.col('cohort_month') == datetime(2010, 12, 1)) & (pl.col('cohort_index') == 11))['retention_rate'][0]
later = retention.filter((pl.col('cohort_month') > datetime(2010, 12, 1)) & (pl.col('cohort_index') == 1))['retention_rate'].mean()
# Calendar effect: is Nov 2011 (the diagonal of the heatmap) higher than the same cohorts' other months?
retention_cal = retention.filter(pl.col('cohort_index') >= 1).with_columns(
    calendar_month=pl.col('cohort_month').dt.offset_by(pl.format('{}mo', pl.col('cohort_index')))
)
nov_cells = retention_cal.filter(pl.col('calendar_month') == datetime(2011, 11, 1))
other_cells = retention_cal.filter((pl.col('calendar_month') < datetime(2011, 11, 1)) & pl.col('cohort_month').is_in(nov_cells['cohort_month'].to_list()))
print(f"Cohorts with a Nov 2011 cell : {nov_cells.height}")
print(f"Avg retention in Nov 2011    : {nov_cells['retention_rate'].mean():.1f}%")
print(f"Avg retention, same cohorts, other months : {other_cells['retention_rate'].mean():.1f}%")

print(f'Takeaway: on average {avg_ret(1):.1f}% of a cohort orders again in month 1, {avg_ret(3):.1f}% in month 3 and {avg_ret(6):.1f}% in month 6.')
print(f'Takeaway: the Dec 2010 cohort keeps {dec10:.1f}% of customers after 11 months, vs month-1 retention of {later:.1f}% for the cohorts after it.')

Cohorts: 12   max period index: 11


cohort_month,cohort_size
str,u32
"""2010-12""",884
"""2011-01""",415
"""2011-02""",380
"""2011-03""",452
"""2011-04""",300
"""2011-05""",284
"""2011-06""",242
"""2011-07""",187
"""2011-08""",169


Cohorts with a Nov 2011 cell : 11
Avg retention in Nov 2011    : 30.5%
Avg retention, same cohorts, other months : 25.3%
Takeaway: on average 21.3% of a cohort orders again in month 1, 24.6% in month 3 and 26.9% in month 6.
Takeaway: the Dec 2010 cohort keeps 50.2% of customers after 11 months, vs month-1 retention of 19.8% for the cohorts after it.


**Result:** On average **21.3%** of a cohort orders again in month 1, 24.6% in month 3 and 26.9% in month 6, so retention drops after the first month and then stays roughly flat instead of decaying. The Dec 2010 cohort is much larger (884 customers vs 169-452 later) and keeps 50.2% at month 11, but part of it is existing customers, because the file starts in Dec 2010 and cannot show their real first order.

The heatmap has a calendar effect: **Nov 2011** (the last diagonal) shows 30.5% average retention vs 25.3% for the same cohorts in other months. Part of the apparent loyalty of older cohorts is the seasonal peak, not customer age. Cohorts after Dec 2010 stay at 15-24% in month 1, so the single-month return rate of new customers is the number to improve.

### 3.6 UK vs non-UK customers

In [24]:
geo_summary = (
    cust_statistic.group_by('is_uk')
    .agg(
        customers=pl.len(),
        net_revenue=pl.col('net_revenue').sum(),
        mean_net_revenue=pl.col('net_revenue').mean(),
        median_net_revenue=pl.col('net_revenue').median(),
        median_orders=pl.col('orders').median(),
        median_aov=pl.col('aov').median(),
        repeat_rate_pct=pl.col('is_repeat').mean() * 100
    )
    .with_columns(
        **{'customer_share_%': pl.col('customers') / pl.col('customers').sum() * 100,
           'revenue_share_%': pl.col('net_revenue') / pl.col('net_revenue').sum() * 100}
    )
    .with_columns(group=pl.when(pl.col('is_uk')).then(pl.lit('UK')).otherwise(pl.lit('Non-UK')))
    .sort('is_uk', descending=True)
)
display(geo_summary.drop('is_uk').with_columns(pl.col(pl.Float64).round(1)))

uk, intl = geo_summary.row(0, named=True), geo_summary.row(1, named=True)
print(f"Takeaway: non-UK customers are {intl['customer_share_%']:.1f}% of customers and {intl['revenue_share_%']:.1f}% of net revenue; "
      f"median AOV {intl['median_aov']:,.0f} vs {uk['median_aov']:,.0f} in the UK.")

customers,net_revenue,mean_net_revenue,median_net_revenue,median_orders,median_aov,repeat_rate_pct,customer_share_%,revenue_share_%,group
u32,f64,f64,f64,f64,f64,f64,f64,f64,str
3905,6.806471e6,1743.0,633.0,2.0,279.2,65.6,90.4,82.3,"""UK"""
415,1463166.3,3525.7,905.3,2.0,387.8,64.1,9.6,17.7,"""Non-UK"""


Takeaway: non-UK customers are 9.6% of customers and 17.7% of net revenue; median AOV 388 vs 279 in the UK.


**Result:** Non-UK customers are **9.6%** of customers (415) and **17.7%** of net revenue. Their median order value is higher (388 vs 279) and their mean net revenue per customer is about twice the UK (3,526 vs 1,743), while their repeat rate is almost the same (64.1% vs 65.6%).

## 4. Statistical Analysis

All tests use `alpha = 0.05` on the customer table (`cust_statistic`), not on invoice lines.

### 4.1 Do UK and non-UK customers differ in average order value?

**H0:** the distribution of customer AOV is the same for UK and non-UK customers.  
**H1:** the distributions differ.

In [25]:
uk_aov = cust_statistic.filter(pl.col('is_uk'))['aov'].to_numpy()
intl_aov = cust_statistic.filter(~pl.col('is_uk'))['aov'].to_numpy()

print(f'Skewness  UK / non-UK : {st.skew(uk_aov):.1f} / {st.skew(intl_aov):.1f}')
print(f'Shapiro p UK / non-UK : {st.shapiro(uk_aov).pvalue:.2e} / {st.shapiro(intl_aov).pvalue:.2e}')

# Strong right skew and Shapiro rejection -> compare distributions with Mann-Whitney, not a t-test
u_stat, p_value = st.mannwhitneyu(uk_aov, intl_aov, alternative='two-sided')
rank_biserial = 2 * u_stat / (len(uk_aov) * len(intl_aov)) - 1

print(f'\nn UK = {len(uk_aov):,}, n non-UK = {len(intl_aov):,}')
print(f'Median AOV UK / non-UK : {np.median(uk_aov):,.0f} / {np.median(intl_aov):,.0f}')
print(f'Mann-Whitney U = {u_stat:,.0f}, p = {p_value:.2e}')
print(f'Rank-biserial effect size = {rank_biserial:+.3f} (positive = UK higher)')
print(f"Conclusion: {'Reject' if p_value < ALPHA else 'Fail to reject'} H0: "
      f"{'UK and non-UK customers have different AOV distributions.' if p_value < ALPHA else 'no evidence that AOV differs by country group.'}")

Skewness  UK / non-UK : 17.1 / 5.0
Shapiro p UK / non-UK : 7.03e-79 / 3.73e-32

n UK = 3,905, n non-UK = 415
Median AOV UK / non-UK : 279 / 388
Mann-Whitney U = 576,506, p = 3.75e-22
Rank-biserial effect size = -0.289 (positive = UK higher)
Conclusion: Reject H0: UK and non-UK customers have different AOV distributions.


> **Result:** Reject H0. Non-UK customers have a higher AOV distribution than UK customers (median 388 vs 279, p = 3.75e-22, rank-biserial effect -0.29, a small-to-medium effect: a random non-UK customer has the higher AOV in roughly 64% of pairs). With 4,320 customers the p-value is tiny for almost any difference, so the effect size is the number that matters. This is an association by country group, not a cause: it can reflect a different customer mix (for example resellers abroad).

### 4.2 Is repeat purchase related to country group?

**H0:** repeat purchase (2+ orders) is independent of UK / non-UK.  
**H1:** repeat rate differs between the groups.

In [26]:
contingency = (
    cust_statistic.group_by('is_uk', 'is_repeat').len()
    .sort('is_uk', 'is_repeat')
    .pivot(on='is_repeat', index='is_uk', values='len')
    .sort('is_uk')
)
table = contingency.select(['false', 'true']).to_numpy()
print('Rows: non-UK, UK | Columns: one-time, repeat')
print(table)

chi2, p_chi, dof, expected = st.chi2_contingency(table)
cramers_v = np.sqrt(chi2 / (table.sum() * (min(table.shape) - 1)))
rates = table[:, 1] / table.sum(axis=1) * 100

print(f'\nRepeat rate non-UK / UK : {rates[0]:.1f}% / {rates[1]:.1f}%   (n = {table.sum(axis=1)[0]:,} / {table.sum(axis=1)[1]:,})')
print(f'Chi-square = {chi2:.2f}, dof = {dof}, p = {p_chi:.3g}, Cramer\'s V = {cramers_v:.3f}')
print(f"Conclusion: {'Reject' if p_chi < ALPHA else 'Fail to reject'} H0: "
      f"{'repeat rate differs between UK and non-UK customers.' if p_chi < ALPHA else 'no evidence that repeat rate depends on country group.'}")

Rows: non-UK, UK | Columns: one-time, repeat
[[ 149  266]
 [1345 2560]]

Repeat rate non-UK / UK : 64.1% / 65.6%   (n = 415 / 3,905)
Chi-square = 0.29, dof = 1, p = 0.589, Cramer's V = 0.008
Conclusion: Fail to reject H0: no evidence that repeat rate depends on country group.


> **Result:** Fail to reject H0. The repeat rate is 64.1% for non-UK and 65.6% for UK customers (p = 0.589, Cramer's V = 0.008, negligible). Country group does not explain who comes back, so retention work does not need to be split by UK vs non-UK.

### 4.3 Which customer metrics move together?

Spearman is used because every metric is right-skewed (section 3.2).

In [27]:
corr_cols = ['orders', 'products', 'aov', 'recency_days', 'net_revenue']
corr_data = cust_statistic.join(rfm.select('customer_id', 'recency_days'), on='customer_id').select(corr_cols)
corr_matrix = st.spearmanr(corr_data.to_numpy()).statistic

fig_corr = px.imshow(
    corr_matrix, x=corr_cols, y=corr_cols, zmin=-1, zmax=1, color_continuous_scale='RdBu_r',
    text_auto='.2f', template='plotly_white',
    title='Spearman correlation between customer metrics',
    labels={'color': 'Spearman rho'}
)
fig_corr.show()

def rho(a, b):
    return corr_matrix[corr_cols.index(a), corr_cols.index(b)]
print(f"Takeaway: orders vs net revenue rho = {rho('orders', 'net_revenue'):+.2f}; products vs net revenue rho = {rho('products', 'net_revenue'):+.2f}; "
      f"recency vs orders rho = {rho('recency_days', 'orders'):+.2f}; aov vs orders rho = {rho('aov', 'orders'):+.2f}.")
print('Correlation is not causation: orders and net revenue are linked by construction, and orders vs recency only shows that active buyers order more.')

Takeaway: orders vs net revenue rho = +0.81; products vs net revenue rho = +0.73; recency vs orders rho = -0.56; aov vs orders rho = +0.16.
Correlation is not causation: orders and net revenue are linked by construction, and orders vs recency only shows that active buyers order more.


**Result:** Number of orders and net revenue move together (rho +0.81), as do distinct products bought and net revenue (+0.73). More recent customers order more often (recency vs orders -0.56). AOV is almost unrelated to the number of orders (+0.16), so order frequency, not basket size, separates high-value customers. These are associations on observational data; orders and revenue are linked by construction, so only a controlled experiment (for example a win-back offer) could show that more orders cause more revenue.

# Final Conclusion

## Key Findings


1. **Revenue is large and seasonal.** Net revenue is **9.77M** over the 13 months (2.3% returned). Sep-Nov hold **37.5%** of the 12 complete months and Nov 2011 is **2.25x** an average Jan-Aug month. The peak comes from **2.03x the orders** at only 1.10x the value per order.
2. **The business is UK-centred but not UK-only.** The UK brings **84.8%** of net revenue. Abroad, the Netherlands, EIRE and Australia depend on 3-9 large accounts (15k-82k per customer), while Germany and France are broad markets with about 2,100 per customer, like the UK.
3. **The catalogue is broad.** The top 10 products make only 8.2% of net revenue; 840 products (21.4%) make 80%.
4. **Customers are heavily concentrated.** The top 10% of customers bring **60.1%** of net revenue; median net revenue per customer is 650 vs a mean of 1,914. Customers with 10+ orders are 8.9% of the base and **52.5%** of revenue.
5. **A third of customers never return.** 34.6% buy once (6.5% of revenue). Average retention is 21.3% in month 1 and 26.9% in month 6, and a share of the older cohorts' retention comes from the Nov 2011 peak (30.5% vs 25.3%).
6. **About 792k of revenue is in customers who went quiet.** `Can't Lose Them` + `At Risk` are 14.9% of customers and 9.6% of net revenue, with 130-164 days since the last order. Champions carry 67.9% of revenue.
7. **Country group does not explain loyalty, but it does explain order size.** Repeat rate is 64.1% vs 65.6% (p = 0.589, V = 0.008); non-UK AOV is higher (388 vs 279, p = 3.75e-22, effect -0.29).
8. **Data quality shapes the numbers.** 24.8% of lines (15.4% of net revenue) have no `customer_id`, and two reversed orders of 77k and 168k would have doubled the apparent return rate (4.6% vs 2.3%).

**Interpretation and assumptions**

-  The weekday-daytime pattern (75.6% of revenue between 10:00 and 15:59, 8.1% on Sunday, no Saturday trade) and the large per-customer values abroad suggest many customers are businesses or resellers.
-  Currency is GBP; the file does not say. Duplicate lines are treated as export errors.
-  The Sep-Nov rise is seasonal, but with one year of data, seasonality cannot be separated from growth. The like-for-like Dec 1-9 window (+9.8%) is only a hint.


## Business Recommendations


**Retention**
- Protect Champions (1,115 customers, 67.9% of revenue): account owner, early access to seasonal ranges, a service-level check before Nov.
- Run a win-back test on `Can't Lose Them` and `At Risk` (642 customers, about 792k net revenue) with a control group, because the data cannot show that an offer would cause a return.
- Test a second-order trigger for one-time buyers (1,494 customers): a timed message or offer inside the first 30 days, since month-1 retention is about 20%.

**Seasonal operations**
- Build stock and staffing plans around Sep-Nov (37.5% of the year) and weekday daytime; Sunday carries 8.1% of revenue and Saturday none.

**International**
- Assign account management to Netherlands, EIRE and Australia (3-9 accounts each). Treat Germany and France as the scalable markets, with about 95 and 87 customers at UK-like value per customer.

**Returns and data**
- Review products with extreme return rates (`PANTRY CHOPPING BOARD` 81.8%, `FAIRY CAKE FLANNEL` 36.5%) for listing or quality problems, and record a return reason.
- Capture `customer_id` at checkout (24.8% of lines are anonymous) and add an alert for single orders above about 50,000 units.